In [1]:
!pip install dicom2nifti --quiet

Note: you may need to restart the kernel to use updated packages.


In [9]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient # Comment out this line if not running in Kaggle
import dicom2nifti
import zipfile
import os

**If You Use Kaggle to Store Secrets**

In [4]:
# Comment out this cell if not running in Kaggle.
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("huggingface_token")
login(hf_token)

In [5]:
from huggingface_hub import hf_hub_download

filepath = hf_hub_download(
    repo_id="deeplearningresearchproject/dataset_project",
    filename="LIDC-IDRI-0050.zip",
    repo_type="dataset"
)

print(filepath)

LIDC-IDRI-0050.zip:   0%|          | 0.00/2.80G [00:00<?, ?B/s]

/root/.cache/huggingface/hub/datasets--deeplearningresearchproject--dataset_project/snapshots/feef574cf9243e5c8692fb6dbf18ac61e780d72a/LIDC-IDRI-0050.zip


In [7]:
class Niftify:

    def __init__(self, zip_path, extract_to, nifti_path):
        self.zip_path = zip_path
        self.extract_to = extract_to
        self.nifti_path = nifti_path
        
    def unzip_file(self):
        """
        Unzips a ZIP file to the specified directory.
    
        :param zip_path: Path to the .zip file
        :param extract_to: Directory where files will be extracted
        """
        try:
            if not os.path.isfile(self.zip_path):
                raise FileNotFoundError(f"ZIP file not found: {self.zip_path}")
    
            os.makedirs(self.extract_to, exist_ok=True)
    
            # Open and extract the ZIP file
            with zipfile.ZipFile(self.zip_path, 'r') as zip_ref:
                zip_ref.extractall(self.extract_to)
                print(f"Extracted {len(zip_ref.namelist())} files to '{self.extract_to}'")
    
        except zipfile.BadZipFile:
            print("Error: The file is not a valid ZIP archive.")
        except PermissionError:
            print("Error: Permission denied while accessing files.")
        except Exception as e:
            print(f"An unexpected error occurred: {e}")

    def to_nifti(self):
        assert os.path.isdir(self.extract_to), f"{self.extract_to} is not a directory"
        
        os.makedirs(self.nifti_path, exist_ok=True)
        dicom2nifti.convert_directory(self.extract_to, self.nifti_path)

    def run(self):
        self.unzip_file()
        self.to_nifti()


In [10]:
dicom_path = './dicom'
nifti_path = './nifti'
niftify = Niftify(filepath, dicom_path, nifti_path)
niftify.run()

Extracted 9142 files to './dicom'


In [11]:
len(os.listdir(nifti_path))

50